In [1]:
!pip install rank-bm25


# 01 Data + BM25 Prep (Phase 1)

Notebook này chạy Task 1.5, 1.6, 1.7 của M1: parse CISI và export `corpus.json`, `queries.json`, `qrels.json`.

In [2]:
from pathlib import Path
import json
import sys

def find_project_root(start: Path) -> Path:
    """Find project root by locating TEAM_ASSIGNMENTS.md and CISI_data."""
    for p in [start, *start.parents]:
        if (p / "TEAM_ASSIGNMENTS.md").exists() and (p / "CISI_data").exists():
            return p
    raise FileNotFoundError("Cannot find project root containing TEAM_ASSIGNMENTS.md and CISI_data")

project_root = find_project_root(Path.cwd().resolve())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.parser import parse_cisi_file, export_qrels_json

data_dir = project_root / "CISI_data"
output_dir = project_root / "data"
output_dir.mkdir(parents=True, exist_ok=True)

all_path = data_dir / "CISI.ALL"
qry_path = data_dir / "CISI.QRY"
rel_path = data_dir / "CISI.REL"
corpus_out = output_dir / "corpus.json"
queries_out = output_dir / "queries.json"
qrels_out = output_dir / "qrels.json"

assert all_path.exists(), f"Missing file: {all_path}"
assert qry_path.exists(), f"Missing file: {qry_path}"
assert rel_path.exists(), f"Missing file: {rel_path}"

print("project_root =", project_root)
print("data_dir =", data_dir)

project_root = C:\Users\Admin\OneDrive - ptit.edu.vn\nam4ky2\Truyxuatthongtin IR\Project
data_dir = C:\Users\Admin\OneDrive - ptit.edu.vn\nam4ky2\Truyxuatthongtin IR\Project\CISI_data


In [3]:
# Parse corpus + queries
docs = parse_cisi_file(all_path)
parsed_queries = parse_cisi_file(qry_path)

# Chuẩn schema theo TEAM_ASSIGNMENTS
corpus = [
    {
        "doc_id": d["doc_id"],
        "title": d["title"],
        "author": d["author"],
        "text": d["text"],
    }
    for d in docs
]

queries = [
    {
        "query_id": q["doc_id"],
        "text": q["text"],
    }
    for q in parsed_queries
]

# Export JSON
corpus_out.write_text(json.dumps(corpus, ensure_ascii=False, indent=2), encoding="utf-8")
queries_out.write_text(json.dumps(queries, ensure_ascii=False, indent=2), encoding="utf-8")
qrels = export_qrels_json(rel_path, qrels_out)

print("Saved:", corpus_out)
print("Saved:", queries_out)
print("Saved:", qrels_out)

Saved: C:\Users\Admin\OneDrive - ptit.edu.vn\nam4ky2\Truyxuatthongtin IR\Project\data\corpus.json
Saved: C:\Users\Admin\OneDrive - ptit.edu.vn\nam4ky2\Truyxuatthongtin IR\Project\data\queries.json
Saved: C:\Users\Admin\OneDrive - ptit.edu.vn\nam4ky2\Truyxuatthongtin IR\Project\data\qrels.json


In [4]:
# Quick validation
assert len(corpus) == 1460, f"Expected 1460 docs, got {len(corpus)}"
assert len(queries) == 112, f"Expected 112 queries, got {len(queries)}"
assert all(set(d.keys()) == {"doc_id", "title", "author", "text"} for d in corpus)
assert all(set(q.keys()) == {"query_id", "text"} for q in queries)

print("Corpus docs:", len(corpus))
print("Queries:", len(queries))
print("Qrels queries:", len(qrels))
print("Sample doc:", corpus[0])
print("Sample query:", queries[0])

Corpus docs: 1460
Queries: 112
Qrels queries: 76
Sample doc: {'doc_id': 1, 'title': '18 Editions of the Dewey Decimal Classifications', 'author': 'Comaromi, J.P.', 'text': "The present study is a history of the DEWEY Decimal Classification. The first edition of the DDC was published in 1876, the eighteenth edition in 1971, and future editions will continue to appear as needed. In spite of the DDC's long and healthy life, however, its full story has never been told. There have been biographies of Dewey that briefly describe his system, but this is the first attempt to provide a detailed history of the work that more than any other has spurred the growth of librarianship in this country and abroad."}
Sample query: {'query_id': 1, 'text': 'What problems and concerns are there in making up descriptive titles? What difficulties are involved in automatically retrieving articles from approximate titles? What is the usual relevance of the content of articles to their titles?'}


## Phase 2 (Task 2.1 - 2.2): Chạy TF-IDF Baseline và Đánh giá (MRR, P@10)


In [5]:
from retrieval.tfidf_baseline import run_tfidf_baseline

print("--- TASK 2.1 & 2.2: TF-IDF Baseline ---")
metrics = run_tfidf_baseline(
    corpus_path=corpus_out,
    queries_path=queries_out,
    qrels_path=qrels_out,
    output_path=project_root / "reports" / "baseline_metrics.json",
    top_k=100
)

print(f"[OK] Saved metrics to reports/baseline_metrics.json")
print(f"[OK] MRR: {metrics['mrr']:.6f}")
print(f"[OK] P@10: {metrics['p@10']:.6f}")


--- TASK 2.1 & 2.2: TF-IDF Baseline ---
[OK] Saved metrics to reports/baseline_metrics.json
[OK] MRR: 0.408398
[OK] P@10: 0.213393


# Phase 2: BM25 Execution & Export (Task 2.5 - 2.9)
Khác với file python, trên Notebook ta trực tiếp khởi tạo index, chạy truy vấn, và xuất kết quả ra file để lưu lại trong môi trường Kaggle.

In [6]:
from retrieval.bm25_retriever import init_default_bm25, bm25_retrieve
import pickle

print("--- TASK 2.5: Build BM25 Index ---")
# Khởi tạo runtime với tham số tối ưu (k1=1.5, b=1.0)
runtime = init_default_bm25(
    corpus_path=corpus_out,
    k1=1.5, 
    b=1.0,
    apply_stemming=False,
    use_stopwords=True
)

print("[OK] BM25 Build Completed.")

print("\n--- TASK 2.8: Serialize Index ---")
index_out_path = output_dir / "bm25_index.pkl"
with open(index_out_path, "wb") as f:
    pickle.dump({"bm25": runtime.bm25, "doc_ids": runtime.doc_ids}, f)
print(f"[OK] Saved BM25 index to {index_out_path}")

print("\n--- TASK 2.9: Retrieve Queries ---")
results_dict = {}
for q in queries:
    qid = str(q["query_id"])
    top_k_res = bm25_retrieve(query=q["text"], top_k=100, runtime=runtime)
    results_dict[qid] = [{"doc_id": doc_id, "score": score} for doc_id, score in top_k_res]

top100_out_path = output_dir / "bm25_top100.json"
with open(top100_out_path, "w", encoding="utf-8") as f:
    json.dump(results_dict, f, indent=2)
print(f"[OK] Saved Top-100 results for {len(queries)} queries to {top100_out_path}")



--- TASK 2.5: Build BM25 Index ---
[OK] BM25 Build Completed.

--- TASK 2.8: Serialize Index ---
[OK] Saved BM25 index to C:\Users\Admin\OneDrive - ptit.edu.vn\nam4ky2\Truyxuatthongtin IR\Project\data\bm25_index.pkl

--- TASK 2.9: Retrieve Queries ---
[OK] Saved Top-100 results for 112 queries to C:\Users\Admin\OneDrive - ptit.edu.vn\nam4ky2\Truyxuatthongtin IR\Project\data\bm25_top100.json
